<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/task3/Task3_Model3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Task 3 (Part 2) — Logistic Regression & Decision Tree

#import and start pyspark
!pip install pyspark -q

from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print('Spark Successfully Started!')

Spark Successfully Started!


In [ ]:
#mount google drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

# Load preprocessed train/test data saved in Task 2
train_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/train_data.parquet'
)
test_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/test_data.parquet'
)

print(f"Training rows: {train_df.count():,}")
print(f"Testing rows: {test_df.count():,}")

# Use a smaller sample for model tuning - due to RAM issue
train_sample = train_df.sample(0.03, seed=42)
test_sample = test_df.sample(0.03, seed=42)

print(f"Training sample rows: {train_sample.count():,}")
print(f"Testing sample rows: {test_sample.count():,}")

print("Data loaded successfully!")

# Need this for ParamGridBuilder used in Models 3 and 4
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

Mounted at /content/drive
Training rows: 1,469,696
Testing rows: 367,299
Training sample rows: 43,955
Testing sample rows: 11,043
Data loaded successfully!


In [ ]:
#MODEL 3: Logistic Regression -
#for classification
#Prediction is the cost is high - yes or no

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

# Define the Logistic Regression classifier
log_Model = LogisticRegression(
    featuresCol='scaled_features',
    labelCol='HIGH_COST',
    maxIter=10,
    family='binomial'
)
# family='binomial' is used because HIGH_COST has two classes:
# 0 = Normal Cost
# 1 = High Cost

print("Logistic Regression classifier defined!")

Logistic Regression classifier defined!


In [ ]:
# HYPERPARAMETER TUNING
# CrossValidator tests different parameter settings and
# automatically selects the model with the highest AUC-ROC score.

# regParam controls regularisation strength.
# Higher values reduce overfitting by penalising large coefficients.

# elasticNetParam controls the type of regularisation:
# 0.0 = Ridge (L2)
# 1.0 = Lasso (L1)
# Values in between combine both approaches.

log_grid = (
    ParamGridBuilder()
    .addGrid(log_Model.regParam, [0.01, 0.1])
    .addGrid(log_Model.elasticNetParam, [0.0, 0.5])
    .build()
)

# AUC-ROC is used as the evaluation metric.
# It measures how well the model separates
# HIGH_COST prescriptions from non-HIGH_COST prescriptions.

# Values closer to 1 indicate stronger classification performance.

log_evaluator = BinaryClassificationEvaluator(
    labelCol='HIGH_COST',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)

# Cross-validation provides a more reliable estimate of model performance by repeatedly training and validating on different subsets of the data.

log_cv = CrossValidator(
    estimator=log_Model,
    estimatorParamMaps=log_grid,
    evaluator=log_evaluator,
    numFolds=2,
    seed=42,          # ensures reproducible results
    parallelism=1     # trains one model at a time to avoid exceeding Google Colab RAM limits.
)

print("Logistic Regression CrossValidator configured!")
print(f"Parameter combinations: {len(log_grid)}")
print(f"Total training runs: {len(log_grid) * 2}")
print("Parameters being tuned:")
print("  regParam: [0.01, 0.1]")
print("  elasticNetParam: [0.0, 0.5]")
print("Evaluation metric: AUC-ROC")
print("Cross-validation folds: 2")

Logistic Regression CrossValidator configured!
Parameter combinations: 4
Total training runs: 8
Parameters being tuned:
  regParam: [0.01, 0.1]
  elasticNetParam: [0.0, 0.5]
Evaluation metric: AUC-ROC
Cross-validation folds: 2


In [ ]:
# TRAINING

print("Training Logistic Regression -")

log_start = time.time()# Record training start time
log_model = log_cv.fit(train_sample) # Train the model and perform cross-validation
log_end = time.time()# Record training end time

# Calculate total training time
log_time = round(log_end - log_start, 2)

print(f"\nTime Taken for Training: {log_time} seconds!")

Training Logistic Regression -

Time Taken for Training: 114.51 seconds!


In [ ]:
# BEST HYPERPARAMETERS
best_log = log_model.bestModel
print(f"Best regParam: {best_log._java_obj.getRegParam()}")
print(f"Best elasticNetParam: {best_log._java_obj.getElasticNetParam()}")

Best regParam: 0.01
Best elasticNetParam: 0.5


In [ ]:
# PREDICTIONS
log_predictions = log_model.transform(test_sample)

# ── MODEL EVALUATION ──
# AUC-ROC measures how well the model distinguishes
# between high-cost and non-high-cost prescriptions.
# Values closer to 1 indicate better classification performance.

log_auc = log_evaluator.evaluate(log_predictions)

# MulticlassClassificationEvaluator is used to calculate
# additional classification metrics.

mc_evaluator = MulticlassClassificationEvaluator(
    labelCol='HIGH_COST',
    predictionCol='prediction'
)

# Accuracy = proportion of correct predictions
log_accuracy = mc_evaluator.evaluate(
    log_predictions,
    {mc_evaluator.metricName: 'accuracy'}
)

# F1 Score balances precision and recall and is particularly useful when class distributions are not perfectly balanced.

log_f1 = mc_evaluator.evaluate(
    log_predictions,
    {mc_evaluator.metricName: 'f1'}
)

# ⭐ ADDED — Precision and Recall (rubric explicitly lists these as acceptable metrics)
log_precision = mc_evaluator.evaluate(
    log_predictions,
    {mc_evaluator.metricName: 'weightedPrecision'}
)
log_recall = mc_evaluator.evaluate(
    log_predictions,
    {mc_evaluator.metricName: 'weightedRecall'}
)

# CONFUSION MATRIX
# Displays the number of correct and incorrect predictions for each class. This helps identify whether the model
# is favouring one class over the other.

print("Confusion Matrix:")
log_predictions.groupBy(
    'HIGH_COST',
    'prediction'
).count().orderBy(
    'HIGH_COST',
    'prediction'
).show()

# FINAL RESULTS
# Display the key evaluation metrics and training time.

print("═" * 45)
print("   LOGISTIC REGRESSION — FINAL RESULTS")
print("═" * 45)
print(f"   AUC-ROC:    {log_auc:.4f}")
print(f"   Accuracy:   {log_accuracy:.4f}")
print(f"   F1 Score:   {log_f1:.4f}")
print(f"   Precision:  {log_precision:.4f}")    # ⭐ ADDED
print(f"   Recall:     {log_recall:.4f}")        # ⭐ ADDED
print(f"   Time:       {log_time} seconds")
print("═" * 45)

Confusion Matrix:
+---------+----------+-----+
|HIGH_COST|prediction|count|
+---------+----------+-----+
|        0|       0.0| 8949|
|        0|       1.0|    1|
|        1|       0.0|  278|
|        1|       1.0| 1815|
+---------+----------+-----+

═════════════════════════════════════════════
   LOGISTIC REGRESSION — FINAL RESULTS
═════════════════════════════════════════════
   AUC-ROC:    0.9995
   Accuracy:   0.9747
   F1 Score:   0.9740
   Precision:  0.9755
   Recall:     0.9747
   Time:       114.51 seconds
═════════════════════════════════════════════


### INTERPRETATION OF RESULTS:

Logistic Regression achieved excellent classification performance
(AUC-ROC = 0.9995, Accuracy = 0.9747, F1 Score = 0.9740) when
predicting whether a prescription exceeds the £50 high-cost threshold.

CONFUSION MATRIX ANALYSIS:

- True Negatives (8,949): Correctly identified low-cost prescriptions
- True Positives (1,815): Correctly identified high-cost prescriptions
- False Positives (1): Incorrectly flagged as high-cost
- False Negatives (278): High-cost prescriptions incorrectly classified
  as low-cost

BUSINESS IMPLICATION:

The model correctly classified 97.47% of prescriptions and produced
only one false positive, demonstrating excellent precision in
identifying genuinely high-cost prescriptions. However, 278 high-cost
prescriptions were missed (false negatives), which could reduce the
effectiveness of NHS cost-monitoring initiatives if expensive
prescriptions are not flagged for review.

From an operational perspective, false negatives are generally more
costly than false positives because they allow potentially expensive
prescriptions to go unnoticed. Therefore, NHS decision-makers may
choose to lower the classification threshold in a production system
to increase detection of high-cost prescriptions, even if this results
in a small increase in false alarms.

The best-performing model used regParam = 0.01 and
elasticNetParam = 0.5, indicating that a combination of L1 and L2
regularisation provided the strongest generalisation performance.
This helped reduce overfitting while maintaining excellent predictive
accuracy on unseen data.

The exceptionally high AUC-ROC score of 0.9995 indicates that the
model can almost perfectly distinguish between high-cost and
low-cost prescriptions. Similar to the Linear Regression results,
this suggests that variables such as NIC are highly informative
predictors of prescription cost behaviour and play a major role in
determining classification outcomes.

In [10]:
#import file for summary table
import json
results = {
    'model': 'Logistic Regression',
    'auc': log_auc, 'accuracy': log_accuracy, 'f1': log_f1, 'time': log_time,
    'regParam': best_log._java_obj.getRegParam(),
    'elasticNet': best_log._java_obj.getElasticNetParam()
}
with open('/content/drive/MyDrive/TASK_DATASET/results_logreg.json', 'w') as f:
    json.dump(results, f)
print("Logistic Regression results saved!")

Logistic Regression results saved!
